In [4]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from PIL import Image
import pandas as pd
from sklearn.model_selection import train_test_split
from torchvision import transforms
from fpdf import FPDF
from tqdm import tqdm


In [5]:
class ChestXrayDataset(Dataset):
    def __init__(self, data, image_dir, tokenizer, transform=None):
        """
        Args:
            data (DataFrame): DataFrame containing the dataset.
            image_dir (str): Directory where the images are stored.
            tokenizer (Tokenizer): Tokenizer used to process the findings text.
            transform (callable, optional): Optional transform to be applied on a sample.
        """
        self.data = data
        self.image_dir = image_dir
        self.tokenizer = tokenizer
        self.transform = transform
        self.resize = transforms.Resize((224, 224))  # Resize images to a fixed size

    def __len__(self):
        return len(self.data)  # The length of the DataFrame

    def __getitem__(self, idx):
        # Get the filename and path for the image
        img_name = self.data.iloc[idx]['filename']  # 'filename' column from projection.csv
        img_path = os.path.join(self.image_dir, img_name)
        
        # Open the image and convert it to RGB
        img = Image.open(img_path).convert("RGB")
        
        # Apply resizing transformation to ensure all images are the same size
        img = self.resize(img)
        
        # Apply other transformations if specified
        if self.transform:
            img = self.transform(img)
        
        # Get the corresponding findings text and tokenize it
        findings_text = str(self.data.iloc[idx]['findings'])  # Ensure it's a string
        findings_input_ids = self.tokenizer.encode(
            findings_text, padding="max_length", truncation=True, max_length=512, return_tensors="pt"
        ).squeeze(0)  # Squeeze to remove unnecessary batch dimension

        # Get the corresponding impression text and tokenize it
        impression_text = str(self.data.iloc[idx]['impression'])  # Ensure it's a string
        impression_input_ids = self.tokenizer.encode(
            impression_text, padding="max_length", truncation=True, max_length=512, return_tensors="pt"
        ).squeeze(0)  # Squeeze to remove unnecessary batch dimension

        # Return the image and tokenized text
        return {'img': img, 'findings_input_ids': findings_input_ids, 'impression_input_ids': impression_input_ids}


In [7]:
def generate_pdf_report(findings, impression, filename="radiology_report.pdf"):
    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()

    pdf.set_font('Arial', 'B', 16)
    pdf.cell(200, 10, txt="Radiology Report", ln=True, align='C')

    pdf.ln(10)  # Line break
    pdf.set_font('Arial', '', 12)
    pdf.multi_cell(0, 10, f"Findings:\n{findings}")
    pdf.ln(5)
    pdf.multi_cell(0, 10, f"Impression:\n{impression}")

    pdf.output(filename)


In [8]:
# Load dataset CSV files
report_df = pd.read_csv("/kaggle/input/chest-xrays-indiana-university/indiana_reports.csv")  # Adjust the path to your report.csv
projection_df = pd.read_csv("/kaggle/input/chest-xrays-indiana-university/indiana_projections.csv")  # Adjust the path to your projection.csv

# Merge data on 'uid' column
merged_data = pd.merge(report_df, projection_df, on="uid")

# Split data into train, validation, and test sets
train_data, test_data = train_test_split(merged_data, test_size=0.2, random_state=42)
train_data, val_data = train_test_split(train_data, test_size=0.125, random_state=42)  # 0.125 of 0.8 = 0.1

# Define image directory path
image_dir = "/kaggle/input/chest-xrays-indiana-university/images/images_normalized"  # Adjust the path to your images directory


In [9]:
tokenizer = AutoTokenizer.from_pretrained("t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("t5-small").to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

2025-08-04 19:57:00.012925: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754337420.352627      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754337420.449987      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [10]:
# Define any additional transformations if necessary
transform = transforms.Compose([
    transforms.ToTensor(),  # Convert images to tensors
])

# Create datasets
train_dataset = ChestXrayDataset(data=train_data, image_dir=image_dir, tokenizer=tokenizer, transform=transform)
val_dataset = ChestXrayDataset(data=val_data, image_dir=image_dir, tokenizer=tokenizer, transform=transform)
test_dataset = ChestXrayDataset(data=test_data, image_dir=image_dir, tokenizer=tokenizer, transform=transform)

# Create DataLoader for train, validation, and test
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)


In [12]:
def train_model(model, train_loader, val_loader, optimizer, epochs=5):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            # Get data from the batch
            images = batch['img'].to(device)
            findings_input_ids = batch['findings_input_ids'].to(device)
            impression_input_ids = batch['impression_input_ids'].to(device)

            # Forward pass
            outputs_f = model(input_ids=findings_input_ids, labels=findings_input_ids)
            outputs_i = model(input_ids=impression_input_ids, labels=impression_input_ids)

            # Calculate loss
            loss_f = outputs_f.loss
            loss_i = outputs_i.loss
            loss = (loss_f + loss_i) / 2

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        # Print average loss for the epoch
        avg_loss = running_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

        # Validation phase
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                images = batch['img'].to(device)
                findings_input_ids = batch['findings_input_ids'].to(device)
                impression_input_ids = batch['impression_input_ids'].to(device)

                outputs_f = model(input_ids=findings_input_ids, labels=findings_input_ids)
                outputs_i = model(input_ids=impression_input_ids, labels=impression_input_ids)

                loss_f = outputs_f.loss
                loss_i = outputs_i.loss
                val_loss += (loss_f + loss_i) / 2

        avg_val_loss = val_loss / len(val_loader)
        print(f"Validation Loss: {avg_val_loss:.4f}")


In [13]:
def inference_and_report(test_loader):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.eval()

    # Pick the first batch from the test loader
    batch = next(iter(test_loader))
    images = batch['img'].to(device)
    findings_input_ids = batch['findings_input_ids'].to(device)
    impression_input_ids = batch['impression_input_ids'].to(device)

    # Generate findings
    outputs_f = model(input_ids=findings_input_ids, labels=findings_input_ids)
    pred_f_ids = outputs_f.logits.argmax(dim=-1)
    pred_f = tokenizer.decode(pred_f_ids[0], skip_special_tokens=True)

    # Generate impression
    outputs_i = model(input_ids=impression_input_ids, labels=impression_input_ids)
    pred_i_ids = outputs_i.logits.argmax(dim=-1)
    pred_i = tokenizer.decode(pred_i_ids[0], skip_special_tokens=True)

    # Save the generated report as a PDF
    generate_pdf_report(pred_f, pred_i, filename="radiology_report.pdf")
    print("PDF report generated: radiology_report.pdf")


In [14]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [15]:
# Step 8: Train the model
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
train_model(model, train_loader, val_loader, optimizer, epochs=5)

Epoch 1/5: 100%|██████████| 654/654 [24:03<00:00,  2.21s/it]


Epoch 1/5, Loss: 0.2702
Validation Loss: 0.0814


Epoch 2/5: 100%|██████████| 654/654 [19:24<00:00,  1.78s/it]


Epoch 2/5, Loss: 0.0189
Validation Loss: 0.0053


Epoch 3/5: 100%|██████████| 654/654 [18:54<00:00,  1.73s/it]


Epoch 3/5, Loss: 0.0031
Validation Loss: 0.0020


Epoch 4/5: 100%|██████████| 654/654 [18:58<00:00,  1.74s/it]


Epoch 4/5, Loss: 0.0014
Validation Loss: 0.0011


Epoch 5/5: 100%|██████████| 654/654 [18:57<00:00,  1.74s/it]


Epoch 5/5, Loss: 0.0008
Validation Loss: 0.0007


In [16]:
# Step 9: Inference on one test image (for generating a report)
inference_and_report(test_loader)

PDF report generated: radiology_report.pdf


In [17]:
# Saving model and optimizer after training
def save_model_and_optimizer(model, optimizer, model_path, optimizer_path):
    # Save model state dictionary
    torch.save(model.state_dict(), model_path)
    # Save optimizer state dictionary
    torch.save(optimizer.state_dict(), optimizer_path)
    print(f"Model and optimizer saved to {model_path} and {optimizer_path}")

# Example usage after training
save_model_and_optimizer(model, optimizer, "model.pth", "optimizer.pth")


Model and optimizer saved to model.pth and optimizer.pth
